In [12]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
%%bash

set -euo pipefail
cd /content
MC_DIR="/content/minecraft"
MC_LOG="$MC_DIR/server.log"
DRIVE_ROOT="/content/drive/MyDrive/MinecraftServer"
DRIVE_SERVER="$DRIVE_ROOT/server"
DRIVE_BACKUPS="$DRIVE_ROOT/backups"
DRIVE_PLAYIT="$DRIVE_ROOT/playit"
PLAYIT_BIN="/content/playit"
PLAYIT_LOG="/content/playit.log"
PLAYIT_CONFIG="/root/.config/playit_gg/playit.toml"
DRIVE_PLAYIT_CONFIG="$DRIVE_PLAYIT/playit.toml"



BACKUP_INTERVAL=600
MAX_BACKUPS=5
MIN_RAM="2G"
MAX_RAM="4G"




PLAYIT_VERSION="1.0.10"
PLAYIT_URL="https://github.com/playit-cloud/playit-agent/releases/download/v${PLAYIT_VERSION}/playit-linux-amd64"
minecraft_running() {
    pgrep -af 'java.*-jar.*server\.jar.*nogui' >/dev/null 2>&1
}
playit_running() {
    pgrep -af '/content/playit' >/dev/null 2>&1
}
port_listening() {
    ss -ltn 2>/dev/null | grep -q ':25565'
}
echo
echo "=========================================="
echo " Made By TheGwimWeeper"
echo "=========================================="
echo
echo "[1/9] Checking Google Drive..."
if [ ! -d "/content/drive/MyDrive" ]; then
    echo
    echo "ERROR: Google Drive is not mounted."
    echo
    echo "Run this in a normal Python cell first:"
    echo
    echo "from google.colab import drive"
    echo "drive.mount('/content/drive')"
    echo
    exit 1
fi
mkdir -p "$DRIVE_ROOT"
mkdir -p "$DRIVE_SERVER"
mkdir -p "$DRIVE_BACKUPS"
mkdir -p "$DRIVE_PLAYIT"
echo "Google Drive is mounted."
echo
echo "[2/9] Checking Playit configuration..."
mkdir -p "$(dirname "$PLAYIT_CONFIG")"
if [ -f "$PLAYIT_CONFIG" ]; then
    echo "Existing Playit configuration found."
    if [ ! -f "$DRIVE_PLAYIT_CONFIG" ]; then
        echo "Saving existing Playit configuration to Drive..."
        cp "$PLAYIT_CONFIG" "$DRIVE_PLAYIT_CONFIG"
        chmod 600 "$DRIVE_PLAYIT_CONFIG"
        echo "Playit configuration saved."
    else
        echo "Drive already contains a Playit configuration."
        echo "Keeping the currently active local configuration."
    fi
elif [ -f "$DRIVE_PLAYIT_CONFIG" ]; then
    echo "No local Playit configuration found."
    echo "Restoring existing configuration from Drive..."
    cp "$DRIVE_PLAYIT_CONFIG" "$PLAYIT_CONFIG"
    chmod 600 "$PLAYIT_CONFIG"
    echo "Existing Playit agent restored."
else
    echo
    echo "WARNING: No existing Playit configuration found."
    echo "If this is your first run, Playit may provide a claim URL."
    echo
fi
echo
echo "[3/9] Installing required packages..."
apt-get update -qq
apt-get install -y -qq \
    openjdk-25-jre-headless \
    wget \
    curl \
    jq \
    rsync
echo
echo "Java:"
java -version
echo
echo "[4/9] Preparing Minecraft..."
mkdir -p "$MC_DIR"
if [ -f "$DRIVE_SERVER/server.jar" ]; then
    echo "Minecraft server found in Google Drive."
    if minecraft_running; then
        echo "Minecraft is already running."
        echo "Skipping restore."
    else
        echo "Restoring Minecraft server from Drive..."
        rsync -a \
            "$DRIVE_SERVER/" \
            "$MC_DIR/"
        echo "Minecraft restored."
    fi
else
    echo "No Minecraft server found in Drive."
    if [ -f "$MC_DIR/server.jar" ]; then
        echo "Existing local Minecraft server found."
        echo "Preserving existing server."
    else
        echo "No local server found."
        echo "Downloading latest official Minecraft server..."
        MANIFEST="https://piston-meta.mojang.com/mc/game/version_manifest_v2.json"
        VERSION=$(curl -fsSL "$MANIFEST" | jq -r '.latest.release')
        if [ -z "$VERSION" ] || [ "$VERSION" = "null" ]; then
            echo "ERROR: Could not determine latest Minecraft version."
            exit 1
        fi
        echo "Latest Minecraft release: $VERSION"
        VERSION_URL=$(curl -fsSL "$MANIFEST" \
            | jq -r --arg VERSION "$VERSION" \
            '.versions[] | select(.id == $VERSION) | .url')
        if [ -z "$VERSION_URL" ] || [ "$VERSION_URL" = "null" ]; then
            echo "ERROR: Could not find version metadata."
            exit 1
        fi
        SERVER_URL=$(curl -fsSL "$VERSION_URL" \
            | jq -r '.downloads.server.url')
        if [ -z "$SERVER_URL" ] || [ "$SERVER_URL" = "null" ]; then
            echo "ERROR: Could not find official server download."
            exit 1
        fi
        wget -q --show-progress \
            "$SERVER_URL" \
            -O "$MC_DIR/server.jar"
        echo
        echo "Minecraft downloaded."
    fi
fi
# Always accept EULA.
printf 'eula=true\n' > "$MC_DIR/eula.txt"
echo
echo "Server JAR:"
ls -lh "$MC_DIR/server.jar"
echo
echo "Synchronizing server with Google Drive..."
rsync -a \
    --exclude="server.log" \
    --exclude="logs/" \
    "$MC_DIR/" \
    "$DRIVE_SERVER/"
echo "Server synchronized."
echo
echo "[5/9] Starting Minecraft..."
if minecraft_running; then
    echo "Minecraft is already running."
else
    cd "$MC_DIR"
    nohup java \
        -Xms"$MIN_RAM" \
        -Xmx"$MAX_RAM" \
        -jar server.jar \
        nogui \
        > "$MC_LOG" 2>&1 &
    MC_PID=$!
    echo "Minecraft PID: $MC_PID"
    echo "Waiting for Minecraft..."
    READY=0
    for i in $(seq 1 120); do
        if grep -q "Done (" "$MC_LOG" 2>/dev/null; then
            READY=1
            break
        fi
        if ! kill -0 "$MC_PID" 2>/dev/null; then
            echo
            echo "ERROR: Minecraft stopped during startup."
            echo
            tail -100 "$MC_LOG"
            exit 1
        fi
        sleep 1
    done
    if [ "$READY" -ne 1 ]; then
        echo
        echo "ERROR: Minecraft did not finish starting."
        echo
        tail -100 "$MC_LOG"
        exit 1
    fi
fi
echo
echo "Minecraft is running."
echo
echo "[6/9] Starting automatic backups..."
cat > /content/minecraft-backup.sh <<'BACKUP_SCRIPT'
#!/bin/bash
set -u
MC_DIR="/content/minecraft"
DRIVE_SERVER="/content/drive/MyDrive/MinecraftServer/server"
DRIVE_BACKUPS="/content/drive/MyDrive/MinecraftServer/backups"
BACKUP_INTERVAL=600
MAX_BACKUPS=5
mkdir -p "$DRIVE_SERVER"
mkdir -p "$DRIVE_BACKUPS"
while true; do
    sleep "$BACKUP_INTERVAL"
    if pgrep -af 'java.*-jar.*server\.jar.*nogui' >/dev/null 2>&1; then
        TIMESTAMP=$(date +"%Y-%m-%d_%H-%M-%S")
        BACKUP_FILE="$DRIVE_BACKUPS/minecraft_$TIMESTAMP.tar.gz"
        echo "[$(date)] Creating backup..."
        # Tell Minecraft to save all chunks.
        # Uses the Minecraft console through its stdin if available.
        if [ -p "$MC_DIR/console.in" ]; then
            echo "save-all flush" > "$MC_DIR/console.in"
            sleep 5
        fi
        # Create compressed backup.
        tar \
            --exclude="./server.log" \
            --exclude="./logs" \
            -czf "$BACKUP_FILE" \
            -C "$MC_DIR" \
            .
        echo "[$(date)] Backup created:"
        echo "$BACKUP_FILE"
        # Update persistent server copy.
        rsync -a \
            --exclude="server.log" \
            --exclude="logs/" \
            "$MC_DIR/" \
            "$DRIVE_SERVER/"
        echo "[$(date)] Drive server copy updated."
        # Keep only newest backups.
        cd "$DRIVE_BACKUPS"
        ls -1t minecraft_*.tar.gz 2>/dev/null \
            | tail -n +$((MAX_BACKUPS + 1)) \
            | xargs -r rm -f
    else
        echo "[$(date)] Minecraft not running. Backup skipped."
    fi
done
BACKUP_SCRIPT
chmod +x /content/minecraft-backup.sh
pkill -f '/content/minecraft-backup.sh' 2>/dev/null || true
nohup bash /content/minecraft-backup.sh \
    > /content/minecraft-backup.log 2>&1 &
BACKUP_PID=$!
echo "Backup PID: $BACKUP_PID"
echo "Backup interval: 10 minutes"
echo "Backups retained: $MAX_BACKUPS"
echo
echo "[7/9] Preparing Playit..."
if [ ! -f "$PLAYIT_BIN" ]; then
    echo "Downloading Playit $PLAYIT_VERSION..."
    wget -q --show-progress \
        "$PLAYIT_URL" \
        -O "$PLAYIT_BIN"
    chmod +x "$PLAYIT_BIN"
else
    echo "Existing Playit binary found."
fi
echo
echo "[8/9] Starting Playit..."
if playit_running; then
    echo "Playit is already running."
else
    if [ -f "$PLAYIT_CONFIG" ]; then
        echo "Existing claimed Playit agent found."
        echo "Starting existing agent..."
    else
        echo
        echo " PLAYIT CLAIM MAY BE REQUIRED"
        echo
        echo "No existing Playit configuration was found."
        echo "If a claim URL appears below, claim the agent."
        echo
    fi
    nohup "$PLAYIT_BIN" \
        > "$PLAYIT_LOG" 2>&1 &
    PLAYIT_PID=$!
    echo "Playit PID: $PLAYIT_PID"
    sleep 8
fi
echo
echo "[9/9] Final status..."
echo
echo " STATUS"
echo
echo "Minecraft:"
if minecraft_running; then
    echo "  RUNNING"
else
    echo "  NOT RUNNING"
fi
echo
echo "Port 25565:"
if port_listening; then
    echo "  LISTENING"
else
    echo "  NOT LISTENING"
fi
echo
echo "Playit:"
if playit_running; then
    echo "  RUNNING"
else
    echo "  NOT RUNNING"
fi
echo
echo "Playit configuration:"
if [ -f "$PLAYIT_CONFIG" ]; then
    echo "  PRESENT"
else
    echo "  NOT FOUND"
fi
echo
echo "Google Drive server:"
if [ -f "$DRIVE_SERVER/server.jar" ]; then
    echo "  PRESENT"
else
    echo "  NOT FOUND"
fi
echo
echo "Automatic backups:"
if pgrep -af '/content/minecraft-backup.sh' >/dev/null 2>&1; then
    echo "  RUNNING"
else
    echo "  NOT RUNNING"
fi
echo
echo " MINECRAFT LOG"
tail -20 "$MC_LOG" 2>/dev/null || true
echo
echo " PLAYIT LOG"
cat "$PLAYIT_LOG" 2>/dev/null || true
echo
echo " BACKUP LOG"
tail -20 /content/minecraft-backup.log 2>/dev/null || true
echo
echo " IMPORTANT"
echo
echo "Live Minecraft:"
echo "  $MC_DIR"
echo
echo "Google Drive:"
echo "  $DRIVE_ROOT"
echo
echo "Backups:"
echo "  $DRIVE_BACKUPS"
echo
echo "Playit config:"
echo "  $PLAYIT_CONFIG"
echo
echo "Public address:"
echo "  Use your existing Playit tunnel address."
echo
echo " SETUP FINISHED"


 Minecraft + Playit.gg + Google Drive

[1/9] Checking Google Drive...
Google Drive is mounted.

[2/9] Checking Playit configuration...
Existing Playit configuration found.
Saving existing Playit configuration to Drive...
Playit configuration saved.

[3/9] Installing required packages...

Java:
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

[4/9] Preparing Minecraft...
No Minecraft server found in Drive.
Existing local Minecraft server found.
Preserving existing server.

Server JAR:
-rw-r--r-- 1 root root 59M Jun 16 12:13 /content/minecraft/server.jar

Synchronizing server with Google Drive...
Server synchronized.

[5/9] Starting Minecraft...
Minecraft PID: 14826
Waiting for Minecraft...

Minecraft is runnin

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "25.0.3" 2026-04-21
OpenJDK Runtime Environment (build 25.0.3+9-2-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 25.0.3+9-2-22.04.2-Ubuntu, mixed mode, sharing)


In [15]:
%%bash

echo "MINECRAFT"
pgrep -af 'java.*-jar.*server.jar.*nogui' || echo "Minecraft: NOT RUNNING"

echo
echo "PORT 25565"
ss -ltnp | grep ':25565' || echo "Port 25565: NOT LISTENING"

echo
echo "PLAYIT"
pgrep -af '/content/playit' || echo "Playit: NOT RUNNING"

echo
echo "GOOGLE DRIVE"
if [ -d "/content/drive/MyDrive/MinecraftServer" ]; then
    echo "Drive: MOUNTED"
else
    echo "Drive: NOT MOUNTED"
fi

echo
echo "SERVER ON DRIVE"
if [ -f "/content/drive/MyDrive/MinecraftServer/server/server.jar" ]; then
    echo "Server backup: PRESENT"
else
    echo "Server backup: MISSING"
fi

echo
echo "PLAYIT CONFIG"
if [ -f "/content/drive/MyDrive/MinecraftServer/playit/playit.toml" ]; then
    echo "Playit config backup: PRESENT"
else
    echo "Playit config backup: MISSING"
fi

echo
echo "BACKUP PROCESS"
pgrep -af '/content/minecraft-backup.sh' || echo "Backup process: NOT RUNNING"

echo
echo "WORLD"
if [ -d "/content/drive/MyDrive/MinecraftServer/server/world" ]; then
    echo "World backup: PRESENT"
else
    echo "World backup: MISSING"
fi

===== MINECRAFT =====
14826 java -Xms2G -Xmx4G -jar server.jar nogui

===== PORT 25565 =====
LISTEN 0      4096               *:25565            *:*    users:(("java",pid=14826,fd=64))       

===== PLAYIT =====
15015 /content/playit

===== GOOGLE DRIVE =====
Drive: MOUNTED

===== SERVER ON DRIVE =====
Server backup: PRESENT

===== PLAYIT CONFIG =====
Playit config backup: PRESENT

===== BACKUP PROCESS =====
15012 bash /content/minecraft-backup.sh

===== WORLD =====
World backup: PRESENT


In [18]:
%%bash
echo "BACKUPS"
ls -lh /content/drive/MyDrive/MinecraftServer/backups/

echo
echo "BACKUP LOG"
tail -20 /content/minecraft-backup.log

BACKUPS
total 370M
-rw------- 1 root root 124M Aug 18 07:27 minecraft_2026-08-18_07-27-24.tar.gz
-rw------- 1 root root 124M Aug 18 07:37 minecraft_2026-08-18_07-37-31.tar.gz
-rw------- 1 root root 124M Aug 18 07:47 minecraft_2026-08-18_07-47-37.tar.gz

BACKUP LOG
[Tue Aug 18 07:27:24 AM UTC 2026] Creating backup...
[Tue Aug 18 07:27:30 AM UTC 2026] Backup created:
/content/drive/MyDrive/MinecraftServer/backups/minecraft_2026-08-18_07-27-24.tar.gz
[Tue Aug 18 07:27:31 AM UTC 2026] Drive server copy updated.
[Tue Aug 18 07:37:31 AM UTC 2026] Creating backup...
[Tue Aug 18 07:37:37 AM UTC 2026] Backup created:
/content/drive/MyDrive/MinecraftServer/backups/minecraft_2026-08-18_07-37-31.tar.gz
[Tue Aug 18 07:37:37 AM UTC 2026] Drive server copy updated.
[Tue Aug 18 07:47:37 AM UTC 2026] Creating backup...
[Tue Aug 18 07:47:43 AM UTC 2026] Backup created:
/content/drive/MyDrive/MinecraftServer/backups/minecraft_2026-08-18_07-47-37.tar.gz
[Tue Aug 18 07:47:43 AM UTC 2026] Drive server copy 